<a href="https://colab.research.google.com/github/jmc929/Modelos1/blob/main/03_Escalado%2BOneHot_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dependencias

In [3]:
import pandas as pd
import numpy as np
import os
import json

# Librerías para el Pipeline de limpieza
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# El modelo seleccionado: Random Forest
from sklearn.ensemble import RandomForestClassifier

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


# Carga de datos desde Kaggle

In [4]:
# 1. Configuración del entorno y descarga de utilidades del curso
!wget --no-cache -O init.py -q https://raw.githubusercontent.com/rramosp/ai4eng.v1/main/content/init.py
import init; init.init(force_download=False); init.get_weblink()

# 2. Configuración de Credenciales de Kaggle
os.environ['KAGGLE_CONFIG_DIR'] = '/content/'
data = {"username":"cmosquera15","key":"ad6af1b3307521c527205d333e396e07"}

with open('kaggle.json','w') as f:
    json.dump(data, f)

!chmod 600 kaggle.json

# 3. Descarga de la competencia y descompresión
!kaggle competitions download -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -q
!unzip -q '*.zip'

# 4. Carga de Dataframes
df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("--- Datos Descargados Exitosamente ---")
print("Train shape:", df.shape)
print("Test shape:", test_df.shape)

# Asignamos a las variables que usa el resto del notebook
df_train = df
df_test = test_df

# Vemos las columnas para saber cuál es el TARGET
print("\nColumnas disponibles:", df_train.columns.tolist())

replicating local resources
--- Datos Descargados Exitosamente ---
Train shape: (692500, 21)
Test shape: (296786, 20)

Columnas disponibles: ['ID', 'PERIODO_ACADEMICO', 'E_PRGM_ACADEMICO', 'E_PRGM_DEPARTAMENTO', 'E_VALORMATRICULAUNIVERSIDAD', 'E_HORASSEMANATRABAJA', 'F_ESTRATOVIVIENDA', 'F_TIENEINTERNET', 'F_EDUCACIONPADRE', 'F_TIENELAVADORA', 'F_TIENEAUTOMOVIL', 'E_PRIVADO_LIBERTAD', 'E_PAGOMATRICULAPROPIO', 'F_TIENECOMPUTADOR', 'F_TIENEINTERNET.1', 'F_EDUCACIONMADRE', 'RENDIMIENTO_GLOBAL', 'INDICADOR_1', 'INDICADOR_2', 'INDICADOR_3', 'INDICADOR_4']


# Configuración de variables


In [5]:
# --- CONFIGURACIÓN DE VARIABLES ---
# Confirmado con tu salida anterior:
TARGET_COL = 'RENDIMIENTO_GLOBAL'  # La columna que vamos a predecir
ID_COL = 'ID'                      # La columna que identifica al estudiante

print(f"Objetivo a predecir: {TARGET_COL}")
print(f"Columna de Identificación: {ID_COL}")

Objetivo a predecir: RENDIMIENTO_GLOBAL
Columna de Identificación: ID


# Separacion de datos


In [6]:
# 1. Separar X (variables) e y (objetivo) del Train
# Verificamos que la columna exista para evitar errores
if TARGET_COL not in df_train.columns:
    raise ValueError(f"¡ERROR CRÍTICO! La columna '{TARGET_COL}' no se encuentra en el DataFrame.")

X = df_train.drop(columns=[TARGET_COL])
y = df_train[TARGET_COL]

# 2. Preparar el Test
X_test_final = df_test.copy()

# Manejo de la columna ID en el Test (Guardarla y borrarla de los datos para predecir)
if ID_COL in X_test_final.columns:
    test_ids = X_test_final[ID_COL]
    X_test_final = X_test_final.drop(columns=[ID_COL])
else:
    test_ids = X_test_final.index

# Manejo de la columna ID en el Train (Borrarla para que coincida con el Test)
if ID_COL in X.columns:
    X = X.drop(columns=[ID_COL])

print(f"Datos listos.")
print(f"Variables para entrenar: {X.shape[1]}")

Datos listos.
Variables para entrenar: 19


# Random forest

In [ ]:
# 1. Identificar columnas numéricas y categóricas automáticamente
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

print(f"Numéricas detectadas: {len(numeric_features)}")
print(f"Categóricas detectadas: {len(categorical_features)}")

# 2. Transformador Numérico: Imputación por MEDIANA
# Usamos Mediana porque es más robusta a valores atípicos. SIN ESCALADO.
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# 3. Transformador Categórico: Imputación + OneHotEncoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 4. Unir todo en el Preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 5. Pipeline Final con RANDOM FOREST
# Usamos n_jobs=-1 para que use todos los núcleos del procesador de Colab y sea más rápido
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

print("Pipeline configurado correctamente.")

Numéricas detectadas: 5
Categóricas detectadas: 14
Pipeline configurado correctamente.


# Entrenamiento

In [2]:
print("Iniciando entrenamiento con el 100% de los datos...")
model.fit(X, y)
print("¡Entrenamiento finalizado exitosamente!")

Iniciando entrenamiento con el 100% de los datos...


NameError: name 'model' is not defined

# GridSearch SVM

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

# Reducir el tamaño del dataset para GridSearchCV
# Ajusta el factor de submuestreo según sea necesario para evitar el error de RAM
sample_size = 0.1 # Usar el 10% de los datos para la búsqueda
X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, test_size=1-sample_size, random_state=RND, stratify=y_train)

lsvc = LinearSVC(max_iter=10000, random_state=RND, dual=False)
param_grid_l = {'C': [0.01, 0.1, 1]}

grid_l = GridSearchCV(lsvc, param_grid_l, cv=3, scoring='accuracy', n_jobs=-1, verbose=2)
t0 = time.time()
grid_l.fit(X_train_small, y_train_small) # Usar la submuestra para la búsqueda
t1 = time.time()
print("GridSearch LinearSVC terminado en {:.2f} min".format((t1-t0)/60))

best_params = grid_l.best_params_
print("Mejores params (LinearSVC):", best_params)

# Entrenar el modelo final con el conjunto de entrenamiento completo y los mejores parámetros
chosen_model = LinearSVC(max_iter=10000, random_state=RND, dual=False, **best_params)
chosen_model.fit(X_train, y_train)

y_pred_val = chosen_model.predict(X_val)
acc_val = accuracy_score(y_val, y_pred_val)
print("Accuracy en validación (LinearSVC):", acc_val)
print(classification_report(y_val, y_pred_val))


Fitting 3 folds for each of 3 candidates, totalling 9 fits
GridSearch LinearSVC terminado en 0.36 min
Mejores params (LinearSVC): {'C': 0.01}


# Evaluación en validación

In [ ]:
y_pred_val = chosen_model.predict(X_val)
acc_val = accuracy_score(y_val, y_pred_val)
print("Accuracy en validación:", acc_val)
print(classification_report(y_val, y_pred_val))

cm = confusion_matrix(y_val, y_pred_val)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicción')
plt.ylabel('Valor real')
plt.title('Matriz de confusión — SVM')
plt.show()

# Guardar y registrar experimento

In [ ]:
joblib.dump(scaler, 'scaler_03_EscaladoOneHot.joblib')
joblib.dump(feature_columns, 'feature_columns_03_EscaladoOneHot.joblib')
joblib.dump(chosen_model, 'svm_chosen_03_EscaladoOneHot.joblib')
print("Guardados: scaler, feature_columns y modelo.")

exp_row = {
    'notebook': '03 - Escalado + OneHot y SVM',
    'model': 'SVM',
    'params': str(best_params) if 'best_params' in locals() else 'NA',
    'val_accuracy': float(acc_val) if 'acc_val' in locals() else None,
    'test_accuracy': None,
    'kaggle_score': None,
    'model_file': 'svm_chosen_03_EscaladoOneHot.joblib',
    'date': datetime.now().isoformat()
}

exp_file = 'experiments.csv'
exp_df = pd.DataFrame([exp_row])
if os.path.exists(exp_file):
    df_exp = pd.read_csv(exp_file)
    df_exp = pd.concat([df_exp, exp_df], ignore_index=True)
else:
    df_exp = exp_df
df_exp.to_csv(exp_file, index=False)
print("Experimento registrado en", exp_file)

# Preparar test_proc, alinear columnas, escalar y crear el CSV de salida

In [ ]:
test_proc = test_proc.copy()

for c in num_cols:
    if c in test_proc.columns:
        med = X_train[c].median() if c in X_train.columns else 0
        test_proc[c] = pd.to_numeric(test_proc[c], errors='coerce').fillna(med)
    else:
        test_proc[c] = X_train[c].median() if c in X_train.columns else 0

for col in feature_columns:
    if col not in test_proc.columns:
        test_proc[col] = 0

extra = set(test_proc.columns) - set(feature_columns)
if extra:
    test_proc = test_proc.drop(columns=list(extra), errors='ignore')

test_proc = test_proc[feature_columns]

if num_cols:
    test_proc[num_cols] = scaler.transform(test_proc[num_cols])

preds_num = chosen_model.predict(test_proc)

inv_map = {0:'bajo', 1:'medio-bajo', 2:'medio-alto', 3:'alto'}
preds_text = [inv_map[int(p)] for p in preds_num]

submission = pd.DataFrame({'ID': test_df['ID'].values, 'RENDIMIENTO_GLOBAL': preds_text})

out_filename = "archivo_EscaladoOneHot_SVM.csv"
submission.to_csv(out_filename, index=False)
print("Archivo creado:", out_filename, " shape:", submission.shape)

# Subir a Kaggle

In [ ]:
!kaggle competitions submit -c udea-ai-4-eng-20252-pruebas-saber-pro-colombia -f archivo_EscaladoOneHot_SVM-RBF.csv -m "03 SVM attempt"